# MCP Card Zip Builder

Given one or more character-list text files (one character name per line, no
header -- the same format used for the card PDF builder), this packages up
the matching `<character>_healthy.jpg` / `<character>_injured.jpg` files into
a zip archive per list.

This notebook is self-contained and doesn't depend on the PDF builder notebook.

## Setup

In [39]:
import zipfile
import re
from pathlib import Path

## Configuration

`CHARACTER_LIST_PATHS` takes one or more list files. Each produces its own zip,
named after the list file (e.g. `avengers.txt` -> `avengers.zip`).

In [40]:

PROJECT_DIR = Path("/Users/brad/Documents/GitHub/crop_mcp_cards/")
CARDS_DIR = PROJECT_DIR / "output" / "2026_august"
OUTPUT_DIR = CARDS_DIR / "zips"

CHARACTER_LIST_PATH = PROJECT_DIR / "input" / "august_2026_all_cards.txt"

## Load a character list

In [41]:
def load_character_names(path: Path):
    """Read a single-column, no-header text file of character names,
    applying the same cleaning used when the image filenames were generated."""
    with open(path) as f:
        raw_names = [line.strip() for line in f if line.strip()]

    cleaned_names = []
    for character_name in raw_names:
        name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', character_name)
        name = re.sub(r'[_\s]+', '_', name)
        name = name.strip('_.')
        cleaned_names.append(name)

    return list(dict.fromkeys(cleaned_names))  # de-dupe, preserve file order

## Match character names to files

In [42]:
def find_character_files(CARDS_DIR: Path, character_names):
    """Find healthy/injured jpgs for the given character names.

    Returns (matched_files, incomplete, not_found):
    - matched_files: list of Paths to include in the zip
    - incomplete: names that only had a healthy OR injured file (still included)
    - not_found: names with no matching files at all (not included)
    """
    healthy, injured = {}, {}
    files = list(CARDS_DIR.glob("*.jpg")) + list(CARDS_DIR.glob("*.jpeg"))
    for f in files:
        stem = f.stem
        if stem.endswith("_healthy"):
            healthy[stem[: -len("_healthy")]] = f
        elif stem.endswith("_injured"):
            injured[stem[: -len("_injured")]] = f

    matched_files, incomplete, not_found = [], [], []
    for name in character_names:
        h, i = healthy.get(name), injured.get(name)
        if h and i:
            matched_files.extend([h, i])
        elif h or i:
            matched_files.append(h or i)
            incomplete.append(name)
        else:
            not_found.append(name)

    return matched_files, incomplete, not_found

## Build a zip for one list

In [43]:
def build_zip(list_path: Path, CARDS_DIR: Path, output_dir: Path):
    names = load_character_names(list_path)
    matched_files, incomplete, not_found = find_character_files(CARDS_DIR, names)

    if not_found:
        print(f"WARNING [{list_path.name}]: {len(not_found)} character(s) with no "
              f"matching files at all:")
        for name in not_found:
            print(f"  - {name}")

    if incomplete:
        print(f"WARNING [{list_path.name}]: {len(incomplete)} character(s) missing "
              f"healthy or injured (included anyway):")
        for name in incomplete:
            print(f"  - {name}")

    output_dir.mkdir(parents=True, exist_ok=True)
    zip_path = output_dir / f"{list_path.stem}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in matched_files:
            zf.write(f, arcname=f.name)

    print(f"-> {zip_path} ({len(matched_files)} files)\n")
    return zip_path

## Run for all configured lists

In [44]:
zip_paths = [build_zip(CHARACTER_LIST_PATH, CARDS_DIR, OUTPUT_DIR)]
print(f"Done: {len(zip_paths)} zip file(s) created in {OUTPUT_DIR}")

-> /Users/brad/Documents/GitHub/crop_mcp_cards/output/2026_august/zips/august_2026_all_cards.zip (458 files)

Done: 1 zip file(s) created in /Users/brad/Documents/GitHub/crop_mcp_cards/output/2026_august/zips
